# IndoMultiDomain-Core V1 — One-Run Builder

Notebook ini membangun **IndoMultiDomain-Core V1 nyata** langsung ke Google Drive.

Yang dilakukan:
1. mount Google Drive;
2. mengunduh sumber terbuka dari Hugging Face dan Mendeley Data melalui API resmi;
3. menyimpan raw source;
4. harmonisasi schema;
5. menghapus kolom identifier langsung;
6. exact deduplication dan source-aware sampling;
7. membuat split train/validation/test;
8. menghasilkan Core V1 + statistik + manifest untuk Hugging Face.

**Tidak ada anotasi semantik baru.** Label asli sumber hanya disimpan sebagai metadata.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip -q install datasets pyarrow openpyxl requests pandas numpy


In [3]:
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/IndoMultiDomain')
ROOT.mkdir(parents=True, exist_ok=True)
print(ROOT)


/content/drive/MyDrive/IndoMultiDomain


## Tulis builder dan source registry


In [4]:
from pathlib import Path
builder_code = '#!/usr/bin/env python3\nfrom __future__ import annotations\nimport os, re, json, hashlib, shutil, zipfile, unicodedata, random, math\nfrom pathlib import Path\nfrom typing import Optional, Iterable\nimport pandas as pd\nimport numpy as np\nimport requests\n\nSEED = 20260915\nrandom.seed(SEED)\nnp.random.seed(SEED)\n\nPII_COL_PATTERNS = [\n    r"(^|_)(user_?name|username|reviewer|reviewer_?name|email|phone|telephone|account_?id|user_?id)($|_)",\n    r"(^|_)(alamat|address)($|_)",\n]\n\nAPP_DOMAIN_MAP = {\n    "kai": ("transportation_mobility","rail_service"),\n    "mobilejkn": ("health_healthcare","public_health_service"),\n    "mobile jkn": ("health_healthcare","public_health_service"),\n    "satusehat": ("health_healthcare","public_health_service"),\n    "jmo": ("government_public_services","social_security_employment"),\n    "bmkg": ("government_public_services","public_information_weather"),\n    "mypertamina": ("government_public_services","energy_public_service"),\n}\n\ndef norm_text(x):\n    if pd.isna(x): return ""\n    x = unicodedata.normalize("NFC", str(x))\n    x = re.sub(r"<[^>]+>", " ", x)\n    x = re.sub(r"\\s+", " ", x).strip()\n    return x\n\ndef text_hash(x):\n    return hashlib.sha256(norm_text(x).encode("utf-8")).hexdigest()\n\ndef infer_column(columns, candidates):\n    norm = {str(c).strip().lower(): c for c in columns}\n    for cand in candidates:\n        if cand.lower() in norm:\n            return norm[cand.lower()]\n    # fuzzy containment fallback\n    for cand in candidates:\n        for low, orig in norm.items():\n            if cand.lower() in low:\n                return orig\n    return None\n\ndef pii_columns(df):\n    drops = []\n    for c in df.columns:\n        low = str(c).lower()\n        if any(re.search(p, low) for p in PII_COL_PATTERNS):\n            drops.append(c)\n    return drops\n\ndef mendeley_list(dataset_id, version, folder_id="root"):\n    url = f"https://data.mendeley.com/public-api/datasets/{dataset_id}/files"\n    r = requests.get(url, params={"folder_id":folder_id, "version":version}, timeout=60)\n    r.raise_for_status()\n    return r.json()\n\ndef mendeley_download_recursive(dataset_id, version, dest: Path):\n    dest.mkdir(parents=True, exist_ok=True)\n    downloaded = []\n    queue = [("root", Path("."))]\n    seen = set()\n    while queue:\n        folder_id, rel = queue.pop(0)\n        if folder_id in seen: \n            continue\n        seen.add(folder_id)\n        items = mendeley_list(dataset_id, version, folder_id)\n        for item in items:\n            name = item.get("name") or item.get("filename") or item.get("id","item")\n            ctype = str(item.get("type","")).lower()\n            is_folder = ctype == "folder" or item.get("content_details") is None\n            if is_folder and item.get("id"):\n                queue.append((item["id"], rel/name))\n                continue\n            cd = item.get("content_details") or {}\n            durl = cd.get("download_url")\n            if not durl:\n                continue\n            out = dest/rel/name\n            out.parent.mkdir(parents=True, exist_ok=True)\n            if not out.exists() or out.stat().st_size == 0:\n                with requests.get(durl, stream=True, timeout=120) as rr:\n                    rr.raise_for_status()\n                    with out.open("wb") as f:\n                        for chunk in rr.iter_content(1024*1024):\n                            if chunk: f.write(chunk)\n            downloaded.append(out)\n    return downloaded\n\ndef unpack_archives(folder: Path):\n    for p in list(folder.rglob("*")):\n        if p.is_file() and p.suffix.lower() == ".zip":\n            d = p.with_suffix("")\n            if not d.exists():\n                d.mkdir(parents=True, exist_ok=True)\n                try:\n                    with zipfile.ZipFile(p) as z:\n                        z.extractall(d)\n                except zipfile.BadZipFile:\n                    pass\n\ndef read_tables(folder: Path):\n    tables = []\n    for p in folder.rglob("*"):\n        if not p.is_file():\n            continue\n        s = p.suffix.lower()\n        try:\n            if s == ".csv":\n                # robust CSV\n                for enc in ("utf-8","utf-8-sig","latin-1"):\n                    try:\n                        df = pd.read_csv(p, encoding=enc, low_memory=False)\n                        break\n                    except Exception:\n                        df = None\n                if df is not None: tables.append((p,df))\n            elif s in (".xlsx",".xls"):\n                xl = pd.ExcelFile(p)\n                for sh in xl.sheet_names:\n                    tables.append((Path(str(p)+f"::{sh}"), pd.read_excel(p, sheet_name=sh)))\n            elif s == ".parquet":\n                tables.append((p,pd.read_parquet(p)))\n            elif s in (".json",".jsonl"):\n                try:\n                    if s == ".jsonl":\n                        tables.append((p,pd.read_json(p, lines=True)))\n                    else:\n                        obj=json.loads(p.read_text(encoding="utf-8"))\n                        if isinstance(obj,list): tables.append((p,pd.DataFrame(obj)))\n                except Exception:\n                    pass\n        except Exception as e:\n            print("READ_FAIL",p,e)\n    return tables\n\ndef choose_best_table(tables, text_candidates, source_id):\n    cand = []\n    for p,df in tables:\n        tc = infer_column(df.columns, text_candidates)\n        if tc is not None:\n            nonempty = df[tc].astype(str).str.strip().ne("").sum()\n            score = nonempty\n            lname = str(p).lower()\n            if source_id=="SRC006" and "raw" in lname: score *= 2\n            if source_id=="SRC008" and "units" in lname: score *= 3\n            if source_id=="SRC003" and "rating" in lname: score *= 2\n            cand.append((score,p,df,tc))\n    if not cand:\n        return None\n    cand.sort(key=lambda x:x[0], reverse=True)\n    return cand[0]\n\ndef sample_stratified(df, target_n, strata_cols):\n    if target_n is None or len(df) <= target_n:\n        return df.copy()\n    strata_cols = [c for c in strata_cols if c in df.columns]\n    if not strata_cols:\n        return df.sample(target_n, random_state=SEED)\n    # proportional with at least one per stratum, deterministic\n    groups = list(df.groupby(strata_cols, dropna=False))\n    total=len(df)\n    pieces=[]\n    remaining=target_n\n    allocations=[]\n    for key,g in groups:\n        n=max(1, round(target_n*len(g)/total))\n        n=min(n,len(g))\n        allocations.append([key,g,n])\n    # adjust total\n    cur=sum(x[2] for x in allocations)\n    while cur>target_n:\n        candidates=[x for x in allocations if x[2]>1]\n        if not candidates: break\n        x=max(candidates,key=lambda z:z[2])\n        x[2]-=1; cur-=1\n    while cur<target_n:\n        candidates=[x for x in allocations if x[2]<len(x[1])]\n        if not candidates: break\n        x=max(candidates,key=lambda z:len(z[1])-z[2])\n        x[2]+=1; cur+=1\n    for _,g,n in allocations:\n        pieces.append(g.sample(n, random_state=SEED))\n    return pd.concat(pieces, ignore_index=True).sample(frac=1,random_state=SEED).reset_index(drop=True)\n\ndef quality_flag(t):\n    t=norm_text(t)\n    if not t: return "empty"\n    if len(t)<10: return "short_text"\n    if "\\ufffd" in t: return "encoding_issue"\n    if len(set(t.lower().split())) <= 2 and len(t.split())>6: return "suspected_spam"\n    return "pass"\n\ndef deterministic_split(h):\n    x=int(h[:8],16)%100\n    return "train" if x<80 else ("validation" if x<90 else "test")\n\ndef harmonize(df, src, text_col, original_id_col=None, source_file=""):\n    df=df.copy()\n    df=df.drop(columns=pii_columns(df), errors="ignore")\n    out=pd.DataFrame(index=df.index)\n    out["text"]=df[text_col].map(norm_text)\n    out=out[out["text"].str.len()>0].copy()\n    # app-aware domain mapping for IGAR\n    base_domain=src["domain"]; base_sub=src["subdomain"]\n    out["domain"]=base_domain\n    out["subdomain"]=base_sub\n    if src["source_id"]=="SRC003":\n        appcol=infer_column(df.columns,["app","app_name","application"])\n        if appcol:\n            apps=df.loc[out.index,appcol].astype(str).str.lower()\n            domains=[]; subs=[]\n            for a in apps:\n                pair=next((v for k,v in APP_DOMAIN_MAP.items() if k in a), (base_domain,base_sub))\n                domains.append(pair[0]); subs.append(pair[1])\n            out["domain"]=domains; out["subdomain"]=subs\n    out["genre"]=src["genre"]\n    out["source_id"]=src["source_id"]\n    out["source_repository"]=src["provider"]\n    out["source_dataset"]=src["name"]\n    out["source_version"]=src["version"]\n    out["source_platform"]=""\n    if original_id_col and original_id_col in df.columns:\n        out["original_id"]=df.loc[out.index,original_id_col].astype(str)\n    else:\n        out["original_id"]=""\n    out["language"]="id"\n    # date/year if discoverable\n    datecol=infer_column(df.columns,["date","timestamp","created_at","at","purchase_date","tanggal"])\n    if datecol:\n        dates=pd.to_datetime(df.loc[out.index,datecol], errors="coerce")\n        out["year"]=dates.dt.year.astype("Int64")\n    else:\n        out["year"]=pd.Series([pd.NA]*len(out), index=out.index, dtype="Int64")\n    # preserve original label/task only\n    labcol=infer_column(df.columns,["sentiment","label","vader_label","labelScoreBase","class"])\n    if labcol:\n        out["original_task"]="source_label"\n        out["original_label"]=df.loc[out.index,labcol].astype(str)\n        out["original_label_type"]=str(labcol)\n    else:\n        out["original_task"]=""\n        out["original_label"]=""\n        out["original_label_type"]=""\n    out["license"]=src["license"]\n    out["provenance_note"]=f\'{src["source_id"]}; source_file={source_file}\'\n    out["char_count"]=out["text"].str.len()\n    out["word_count"]=out["text"].str.split().str.len()\n    out["quality_flag"]=out["text"].map(quality_flag)\n    out["duplicate_group"]=out["text"].map(text_hash)\n    out["split"]=out["duplicate_group"].map(deterministic_split)\n    codes = {\n      "education":"EDU","health_healthcare":"HLT","finance_banking":"FIN",\n      "government_public_services":"GOV","ecommerce_retail":"ECO",\n      "transportation_mobility":"TRN","tourism_hospitality":"TOU",\n      "law_governance":"LAW"\n    }\n    seq=np.arange(1,len(out)+1)\n    out["imd_id"]=[f\'IMD-{codes.get(d,"OTH")}-{src["source_id"]}-{i:08d}\' for d,i in zip(out["domain"],seq)]\n    cols=["imd_id","text","domain","subdomain","genre","source_id","source_repository",\n          "source_dataset","source_version","source_platform","original_id","language","year",\n          "original_task","original_label","original_label_type","license","provenance_note",\n          "char_count","word_count","quality_flag","duplicate_group","split"]\n    return out[cols].reset_index(drop=True)\n\ndef load_hf_bse(src, raw_dir):\n    from datasets import load_dataset\n    ds=load_dataset(src["identifier"])\n    df=ds["train"].to_pandas()\n    raw_file=raw_dir/f\'{src["source_id"]}_bse_raw.parquet\'\n    df.to_parquet(raw_file,index=False)\n    return df, raw_file\n\ndef process_source(src, root):\n    raw=root/"02_raw_sources"/src["source_id"]\n    harm=root/"03_harmonized"/"source_level"\n    raw.mkdir(parents=True,exist_ok=True); harm.mkdir(parents=True,exist_ok=True)\n    if src["provider"]=="huggingface":\n        df, source_file = load_hf_bse(src, raw)\n        tc=infer_column(df.columns,src["text_candidates"].split("|"))\n        ic=infer_column(df.columns,src["id_candidates"].split("|")) if src["id_candidates"] else None\n    else:\n        version=str(src["version"])\n        try:\n            mendeley_download_recursive(src["identifier"],version,raw)\n        except Exception as e:\n            if src["source_id"]=="SRC002" and version!="1":\n                print("Retry telemedicine with v1:",e)\n                version="1"\n                src=dict(src);src["version"]="1"\n                mendeley_download_recursive(src["identifier"],version,raw)\n            else:\n                raise\n        unpack_archives(raw)\n        tables=read_tables(raw)\n        best=choose_best_table(tables,src["text_candidates"].split("|"),src["source_id"])\n        if best is None:\n            print("No suitable text table for",src["source_id"])\n            for p,df0 in tables[:10]:\n                print(" ",p, list(df0.columns), len(df0))\n            return None\n        _,source_file,df,tc=best\n        ic=infer_column(df.columns,src["id_candidates"].split("|")) if src["id_candidates"] else None\n\n        # Travel: aggressively avoid ground-truth/reference summary tables\n        if src["source_id"]=="SRC007":\n            lname=str(source_file).lower()\n            if any(k in lname for k in ("ground","summary","reference")):\n                alternatives=[]\n                for p,d in tables:\n                    tc2=infer_column(d.columns,src["text_candidates"].split("|"))\n                    if tc2 and not any(k in str(p).lower() for k in ("ground","summary","reference")):\n                        alternatives.append((len(d),p,d,tc2))\n                if alternatives:\n                    alternatives.sort(reverse=True,key=lambda x:x[0])\n                    _,source_file,df,tc=alternatives[0]\n\n    # dedup exact before sampling\n    work=df.copy()\n    work["_tmp_text"]=work[tc].map(norm_text)\n    work=work[work["_tmp_text"].str.len()>0].copy()\n    work["_tmp_hash"]=work["_tmp_text"].map(text_hash)\n    work=work.drop_duplicates("_tmp_hash",keep="first").drop(columns=["_tmp_text","_tmp_hash"])\n    # infer strata\n    strata=[]\n    for cand in ["app","app_name","rating","score","product_id","category","mata_pelajaran","kelas","year","type","jenis"]:\n        c=infer_column(work.columns,[cand])\n        if c and c not in strata: strata.append(c)\n    work=sample_stratified(work,src["target_n"],strata[:3])\n    out=harmonize(work,src,tc,ic,str(source_file))\n    out=out[out["quality_flag"]!="empty"].copy()\n    outpath=harm/f\'{src["source_id"]}_harmonized.parquet\'\n    out.to_parquet(outpath,index=False)\n    out.to_csv(harm/f\'{src["source_id"]}_harmonized.csv\',index=False)\n    print(src["source_id"],"raw",len(df),"selected",len(work),"core",len(out),"file",source_file)\n    return out\n\ndef build_all(root: Path, sources):\n    for d in ["01_registry","02_raw_sources","03_harmonized/source_level","04_analysis","05_manuscript","06_builder","07_huggingface_release"]:\n        (root/d).mkdir(parents=True,exist_ok=True)\n    parts=[]\n    failures=[]\n    for src in sources:\n        if not str(src["status"]).startswith("ACCEPT"):\n            continue\n        print("\\n==",src["source_id"],src["name"],"==")\n        try:\n            out=process_source(dict(src),root)\n            if out is not None: parts.append(out)\n        except Exception as e:\n            print("FAILED",src["source_id"],repr(e))\n            failures.append({"source_id":src["source_id"],"error":repr(e)})\n    if not parts:\n        raise RuntimeError("No source completed.")\n    core=pd.concat(parts,ignore_index=True)\n    # Cross-source exact duplicate grouping; keep canonical first but preserve report\n    core["duplicate_group"]=core["text"].map(text_hash)\n    dup_report=core[core.duplicated("duplicate_group",keep=False)].sort_values("duplicate_group")\n    dup_report.to_csv(root/"04_analysis"/"exact_duplicate_report.csv",index=False)\n    before=len(core)\n    core=core.drop_duplicates("duplicate_group",keep="first").reset_index(drop=True)\n    # regenerate splits after global dedup\n    core["split"]=core["duplicate_group"].map(deterministic_split)\n    core.to_parquet(root/"03_harmonized"/"indomultidomain_core_v1.parquet",index=False)\n    core.to_csv(root/"03_harmonized"/"indomultidomain_core_v1.csv",index=False)\n    for sp in ["train","validation","test"]:\n        core[core["split"]==sp].to_parquet(root/"03_harmonized"/f"indomultidomain_{sp}_v1.parquet",index=False)\n\n    # statistics\n    source_stats=core.groupby(["source_id","source_dataset"],dropna=False).agg(\n        records=("imd_id","count"),words=("word_count","sum"),chars=("char_count","sum"),\n        median_words=("word_count","median")\n    ).reset_index()\n    domain_stats=core.groupby(["domain"],dropna=False).agg(\n        records=("imd_id","count"),words=("word_count","sum"),chars=("char_count","sum"),\n        median_words=("word_count","median"),sources=("source_id","nunique"),genres=("genre","nunique")\n    ).reset_index()\n    genre_stats=core.groupby(["genre"],dropna=False).agg(\n        records=("imd_id","count"),words=("word_count","sum"),chars=("char_count","sum"),\n        median_words=("word_count","median")\n    ).reset_index()\n    quality_stats=core.groupby(["quality_flag"],dropna=False).size().reset_index(name="records")\n    split_stats=core.groupby(["split"],dropna=False).size().reset_index(name="records")\n    source_stats.to_csv(root/"04_analysis"/"source_statistics.csv",index=False)\n    domain_stats.to_csv(root/"04_analysis"/"domain_statistics.csv",index=False)\n    genre_stats.to_csv(root/"04_analysis"/"genre_statistics.csv",index=False)\n    quality_stats.to_csv(root/"04_analysis"/"quality_statistics.csv",index=False)\n    split_stats.to_csv(root/"04_analysis"/"split_statistics.csv",index=False)\n\n    summary={\n        "dataset":"IndoMultiDomain-Core V1",\n        "records":int(len(core)),\n        "records_before_global_exact_dedup":int(before),\n        "domains":int(core.domain.nunique()),\n        "genres":int(core.genre.nunique()),\n        "sources":int(core.source_id.nunique()),\n        "words":int(core.word_count.sum()),\n        "characters":int(core.char_count.sum()),\n        "failures":failures,\n    }\n    (root/"04_analysis"/"build_summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")\n\n    # release manifest + dataset card starter\n    pd.DataFrame(sources).to_csv(root/"07_huggingface_release"/"source_manifest.csv",index=False)\n    card=f"""---\nlanguage:\n- id\nlicense: other\npretty_name: IndoMultiDomain-Core V1\n---\n\n# IndoMultiDomain-Core V1\n\nA provenance-aware, harmonized, multi-domain corpus of natural Indonesian digital text.\n\n## Build summary\n\n- Records: {summary[\'records\']:,}\n- Domains: {summary[\'domains\']}\n- Genres: {summary[\'genres\']}\n- Source datasets included: {summary[\'sources\']}\n- Words: {summary[\'words\']:,}\n\nThe release is distributed under source-specific licenses. See `source_manifest.csv`.\nNo new semantic annotation is introduced by IndoMultiDomain V1; original labels are retained only as provenance metadata.\n\n## Core principles\n- natural/non-synthetic text\n- explicit source provenance\n- source-aware controlled sampling\n- minimal reversible cleaning\n- removal of direct identifier columns\n- exact duplicate control\n- grouped deterministic train/validation/test splits\n"""\n    (root/"07_huggingface_release"/"README.md").write_text(card,encoding="utf-8")\n    print("\\nFINAL SUMMARY")\n    print(json.dumps(summary,ensure_ascii=False,indent=2))\n    return core,summary\n\n'
(ROOT/'06_builder').mkdir(parents=True, exist_ok=True)
(ROOT/'06_builder'/'indomultidomain_build_v1.py').write_text(builder_code, encoding='utf-8')
print('builder written')


builder written


In [5]:
import pandas as pd
SOURCES = [{'source_id': 'SRC001', 'name': 'Indo-Bloom BSE RAW Corpus', 'provider': 'huggingface', 'identifier': 'Firmansyah-Ibrahim/indo-bloom-raw-bse', 'version': 'main', 'domain': 'education', 'subdomain': 'school_textbook', 'genre': 'textbook_passage', 'license': 'CC BY 4.0', 'target_n': None, 'status': 'ACCEPT', 'text_candidates': 'context|context_raw', 'id_candidates': 'chunk_id', 'notes': 'Use cleaned context as Core text; preserve subject/grade/page metadata.'}, {'source_id': 'SRC002', 'name': 'Indonesian Telemedicine User Review Dataset', 'provider': 'mendeley', 'identifier': 'rmmmf2dp8d', 'version': '2', 'domain': 'health_healthcare', 'subdomain': 'telemedicine', 'genre': 'user_review', 'license': 'CC BY 4.0', 'target_n': 12000, 'status': 'ACCEPT_SAMPLE', 'text_candidates': 'review|content|text', 'id_candidates': '', 'notes': 'Alodokter, Halodoc, KlikDokter. If v2 unavailable, notebook falls back to v1.'}, {'source_id': 'SRC003', 'name': 'IGAR Indonesian Government Application Reviews', 'provider': 'mendeley', 'identifier': '7zryc6k76z', 'version': '3', 'domain': 'government_public_services', 'subdomain': 'source_entity_mapped', 'genre': 'user_review', 'license': 'CC BY 4.0', 'target_n': 15000, 'status': 'ACCEPT_SAMPLE', 'text_candidates': 'Content|content|review|text', 'id_candidates': '', 'notes': 'App-aware mapping: KAI transport; MobileJKN/Satusehat health; JMO social security/employment; BMKG public information; MyPertamina energy/public service.'}, {'source_id': 'SRC004', 'name': 'BCA Mobile Reviews', 'provider': 'mendeley', 'identifier': 'kgr7x9vs9v', 'version': '1', 'domain': 'finance_banking', 'subdomain': 'mobile_banking', 'genre': 'user_review', 'license': 'CC BY 4.0', 'target_n': 5000, 'status': 'ACCEPT_SAMPLE', 'text_candidates': 'content|review|text', 'id_candidates': '', 'notes': '26,261 Indonesian reviews; reviewer names excluded.'}, {'source_id': 'SRC005', 'name': 'Livin by Mandiri Reviews', 'provider': 'mendeley', 'identifier': 'h8p5v6r6dn', 'version': '1', 'domain': 'finance_banking', 'subdomain': 'mobile_banking', 'genre': 'user_review', 'license': 'CC BY 4.0', 'target_n': 4000, 'status': 'ACCEPT_SAMPLE', 'text_candidates': 'content|review|text', 'id_candidates': '', 'notes': '10,000 Indonesian reviews; reviewer names excluded.'}, {'source_id': 'SRC006', 'name': 'Dataset Customer Review Indonesia', 'provider': 'mendeley', 'identifier': '74xrbd4vxy', 'version': '1', 'domain': 'ecommerce_retail', 'subdomain': 'product_review', 'genre': 'user_review', 'license': 'CC BY 4.0', 'target_n': 15000, 'status': 'ACCEPT_SAMPLE', 'text_candidates': 'review|raw_review|text|content', 'id_candidates': 'review_id|id', 'notes': '203,786 authentic Indonesian e-commerce reviews; prefer raw dataset file.'}, {'source_id': 'SRC007', 'name': 'Indonesian Travel Reviews for Text Summarization', 'provider': 'mendeley', 'identifier': 'x2r86kfrhp', 'version': '1', 'domain': 'tourism_hospitality', 'subdomain': 'attraction_hotel_restaurant', 'genre': 'user_review', 'license': 'CC BY 4.0', 'target_n': 1500, 'status': 'ACCEPT', 'text_candidates': 'review|text|content', 'id_candidates': '', 'notes': 'Use five natural reviews per object; exclude expert ground-truth summaries from Core.'}, {'source_id': 'SRC008', 'name': 'Indonesian Legislation as Data', 'provider': 'mendeley', 'identifier': '8wpjvhh7g2', 'version': '1', 'domain': 'law_governance', 'subdomain': 'legislation', 'genre': 'legal_norm', 'license': 'CC BY 4.0', 'target_n': 15000, 'status': 'ACCEPT_SAMPLE', 'text_candidates': 'text_normalized|normalized_text|text|unit_text|original_text', 'id_candidates': 'unit_id|id', 'notes': '90,588 norm units with page/line provenance. Prefer units.csv.'}, {'source_id': 'SRC009', 'name': 'Grab App Reviews Indonesia', 'provider': 'mendeley', 'identifier': 'swf47j26v8', 'version': '1', 'domain': 'transportation_mobility', 'subdomain': 'ride_hailing', 'genre': 'user_review', 'license': 'CC BY 4.0', 'target_n': None, 'status': 'ACCEPT', 'text_candidates': 'review|content|text', 'id_candidates': '', 'notes': 'Small transport source; include all valid reviews.'}]
(ROOT/'01_registry').mkdir(parents=True, exist_ok=True)
pd.DataFrame(SOURCES).to_csv(ROOT/'01_registry'/'source_registry_v1_1.csv', index=False)
pd.DataFrame(SOURCES)[['source_id','name','provider','identifier','version','domain','target_n','license','status']]


,source_id,name,provider,identifier,version,domain,target_n,license,status
0,SRC001,Indo-Bloom BSE RAW Corpus,huggingface,Firmansyah-Ibrahim/indo-bloom-raw-bse,main,education,NaN,CC BY 4.0,ACCEPT
1,SRC002,Indonesian Telemedicine User Review Dataset,mendeley,rmmmf2dp8d,2,health_healthcare,12000.0,CC BY 4.0,ACCEPT_SAMPLE
2,SRC003,IGAR Indonesian Government Application Reviews,mendeley,7zryc6k76z,3,government_public_services,15000.0,CC BY 4.0,ACCEPT_SAMPLE
3,SRC004,BCA Mobile Reviews,mendeley,kgr7x9vs9v,1,finance_banking,5000.0,CC BY 4.0,ACCEPT_SAMPLE
4,SRC005,Livin by Mandiri Reviews,mendeley,h8p5v6r6dn,1,finance_banking,4000.0,CC BY 4.0,ACCEPT_SAMPLE
5,SRC006,Dataset Customer Review Indonesia,mendeley,74xrbd4vxy,1,ecommerce_retail,15000.0,CC BY 4.0,ACCEPT_SAMPLE
6,SRC007,Indonesian Travel Reviews for Text Summarization,mendeley,x2r86kfrhp,1,tourism_hospitality,1500.0,CC BY 4.0,ACCEPT
7,SRC008,Indonesian Legislation as Data,mendeley,8wpjvhh7g2,1,law_governance,15000.0,CC BY 4.0,ACCEPT_SAMPLE
8,SRC009,Grab App Reviews Indonesia,mendeley,swf47j26v8,1,transportation_mobility,NaN,CC BY 4.0,ACCEPT


## Jalankan seluruh proses

Cell ini dapat memerlukan waktu karena IGAR dan beberapa dataset Mendeley cukup besar. Jangan hentikan Colab selama cell berjalan. Semua file yang selesai diunduh tetap tersimpan di Drive, sehingga restart dapat melanjutkan tanpa mengunduh ulang file yang sudah ada.


In [6]:
import sys, importlib.util
builder_path = ROOT/'06_builder'/'indomultidomain_build_v1.py'
spec = importlib.util.spec_from_file_location('imd_builder', builder_path)
imd = importlib.util.module_from_spec(spec)
spec.loader.exec_module(imd)
core, summary = imd.build_all(ROOT, SOURCES)



== SRC001 Indo-Bloom BSE RAW Corpus ==


README.md:   0%|          | 0.00/14.1k [00:00<?, ?B/s]

IndoBloom_Master_RAW_BSE_Corpus.csv:   0%|          | 0.00/4.56M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1825 [00:00<?, ? examples/s]

/content/drive/MyDrive/IndoMultiDomain/06_builder/indomultidomain_build_v1.py:239: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates=pd.to_datetime(df.loc[out.index,datecol], errors="coerce")


SRC001 raw 1825 selected 1825 core 1825 file /content/drive/MyDrive/IndoMultiDomain/02_raw_sources/SRC001/SRC001_bse_raw.parquet

== SRC002 Indonesian Telemedicine User Review Dataset ==
Retry telemedicine with v1: 403 Client Error: Forbidden for url: https://data.mendeley.com/public-api/datasets/rmmmf2dp8d/files?folder_id=root&version=2
FAILED SRC002 HTTPError('403 Client Error: Forbidden for url: https://data.mendeley.com/public-api/datasets/rmmmf2dp8d/files?folder_id=root&version=1')

== SRC003 IGAR Indonesian Government Application Reviews ==
FAILED SRC003 HTTPError('403 Client Error: Forbidden for url: https://data.mendeley.com/public-api/datasets/7zryc6k76z/files?folder_id=root&version=3')

== SRC004 BCA Mobile Reviews ==
FAILED SRC004 HTTPError('403 Client Error: Forbidden for url: https://data.mendeley.com/public-api/datasets/kgr7x9vs9v/files?folder_id=root&version=1')

== SRC005 Livin by Mandiri Reviews ==
FAILED SRC005 HTTPError('403 Client Error: Forbidden for url: https://d

## Verifikasi hasil


In [7]:
import json, pandas as pd
print(json.dumps(summary, ensure_ascii=False, indent=2))
display(pd.read_csv(ROOT/'04_analysis'/'domain_statistics.csv'))
display(pd.read_csv(ROOT/'04_analysis'/'source_statistics.csv'))
print('Core parquet:', ROOT/'03_harmonized'/'indomultidomain_core_v1.parquet')
print('HF release:', ROOT/'07_huggingface_release')


{
  "dataset": "IndoMultiDomain-Core V1",
  "records": 1825,
  "records_before_global_exact_dedup": 1825,
  "domains": 1,
  "genres": 1,
  "sources": 1,
  "words": 279884,
  "characters": 2125689,
  "failures": [
    {
      "source_id": "SRC002",
      "error": "HTTPError('403 Client Error: Forbidden for url: https://data.mendeley.com/public-api/datasets/rmmmf2dp8d/files?folder_id=root&version=1')"
    },
    {
      "source_id": "SRC003",
      "error": "HTTPError('403 Client Error: Forbidden for url: https://data.mendeley.com/public-api/datasets/7zryc6k76z/files?folder_id=root&version=3')"
    },
    {
      "source_id": "SRC004",
      "error": "HTTPError('403 Client Error: Forbidden for url: https://data.mendeley.com/public-api/datasets/kgr7x9vs9v/files?folder_id=root&version=1')"
    },
    {
      "source_id": "SRC005",
      "error": "HTTPError('403 Client Error: Forbidden for url: https://data.mendeley.com/public-api/datasets/h8p5v6r6dn/files?folder_id=root&version=1')"
    },

,domain,records,words,chars,median_words,sources,genres
0,education,1825,279884,2125689,154.0,1,1


,source_id,source_dataset,records,words,chars,median_words
0,SRC001,Indo-Bloom BSE RAW Corpus,1825,279884,2125689,154.0


Core parquet: /content/drive/MyDrive/IndoMultiDomain/03_harmonized/indomultidomain_core_v1.parquet
HF release: /content/drive/MyDrive/IndoMultiDomain/07_huggingface_release


In [8]:
# Sanity checks
assert len(core) > 0
assert core['imd_id'].is_unique
assert core['duplicate_group'].is_unique
assert set(core['split']).issubset({'train','validation','test'})
assert core['text'].str.len().gt(0).all()
print('SANITY CHECKS: PASS')


SANITY CHECKS: PASS
